# Proyecto PLN — Clasificación de Textos: Poesía vs Rap
## Inteligencia Artificial Avanzada

**Objetivo:** Construir tres clasificadores neuronales (PAD/Bengio, RNN Elman, Transformer)  
para distinguir textos de **poesía** (clase P) de textos de **rap** (clase R).

| Parte | Descripción |
|-------|-------------|
| 1 | Corpus de trabajo |
| 2 | Preprocesado, tokenización, vocabulario y conjunto de entrenamiento |
| 3 | Modelos neuronales: PAD (Bengio), RNN (Elman), Transformer |
| 4 | Clasificación y generación del CSV de resultados |

---
**Nota sobre el código fuente:**  
Los modelos PAD y RNN utilizan **estrictamente** el código proporcionado en las  
prácticas A83 (Bengio clasificador) y A84 (Elman RNN clasificador).  
El Transformer se implementa con las mismas primitivas de PyTorch (`torch.nn`).

## 0. Instalación e Importaciones

In [ ]:
# Instalamos spaCy para el preprocesado (mejor rendimiento en español)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import re
import random
import math
import csv
import os
from collections import Counter
from functools import partial
import spacy

# Cargamos el modelo de spaCy en español
nlp = spacy.load('es_core_news_sm', disable=['parser', 'ner'])

print(f'PyTorch versión: {torch.__version__}')
print(f'GPU disponible:  {torch.cuda.is_available()}')
print('OK - Librerías importadas correctamente.')

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps') # Aceleración nativa para Mac M1/M2/M3
else:
    device = torch.device('cpu')
print(f'Dispositivo de entrenamiento: {device}')

---
## PARTE 1: Corpus de trabajo

Los corpus siguen el formato definido en el proyecto:  
- Cada texto delimitado por `<CS>` y `</CS>`
- Los saltos de línea representados con `<EOL>`

**Actualiza las rutas a tus corpus antes de ejecutar.**

In [ ]:
# ── Rutas a los corpus ────────────────────────────────────────────────────────
# Sube tus archivos a Google Colab y actualiza las rutas:
PATH_CORPUS_POESIA = './corpus_poesia.txt'   # Clase P (poesía)
PATH_CORPUS_RAP    = './corpus_rap.txt'       # Clase R (rap)

# Directorio de salida para los modelos guardados
OUTPUT_DIR = './modelos_guardados'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Rutas configuradas.')
print(f'  Corpus poesía : {PATH_CORPUS_POESIA}')
print(f'  Corpus rap    : {PATH_CORPUS_RAP}')
print(f'  Modelos en    : {OUTPUT_DIR}')

---
## PARTE 2: Preprocesado, Tokenización, Vocabulario y Dataset

### 2.1 Lectura y Tokenización del Corpus

*(Código de prácticas A81/A82/A83/A84 — sin cambios)*

In [ ]:
import re

def read_and_tokenize(filename):
    """
    Lee un corpus de canciones con formato <CS>...</CS> y lo tokeniza.
    Tokens especiales preservados: <CS>, </CS>, <EOL>
    El resto de puntuación se separa en tokens individuales.
    CAMBIO respecto a la práctica: detección automática de codificación
    (utf-8-sig / utf-16 / latin-1). La lógica de tokenización es idéntica.
    """
    content = None
    for enc in ('utf-8-sig', 'utf-16', 'latin-1'):
        try:
            with open(filename, 'r', encoding=enc) as f:
                content = f.read()
            break
        except (UnicodeDecodeError, UnicodeError):
            continue
    if content is None:
        raise ValueError(f'No se pudo leer {filename}')

    tokens = re.findall(r'<CS>|</CS>|<EOL>|\w+|[^\w\s<>/]', content)

    sentences = []
    current_song = []

    for token in tokens:
        if token == '<CS>':
            if current_song:
                sentences.append(current_song)
            current_song = []
        elif token == '</CS>':
            if current_song:
                sentences.append(current_song)
            current_song = []
        else:
            current_song.append(token)

    if current_song:
        sentences.append(current_song)

    return sentences


# ── Carga de los corpus ───────────────────────────────────────────────────────
PATH_CORPUS_POESIA = './corpus_poesia.txt'
PATH_CORPUS_RAP    = './corpus_rap.txt'

sentences_poesia = read_and_tokenize(PATH_CORPUS_POESIA)
sentences_rap    = read_and_tokenize(PATH_CORPUS_RAP)

print(f'Corpus poesía: {len(sentences_poesia)} textos')
print(f'Corpus rap   : {len(sentences_rap)} textos')
print(f'\nEjemplo poesía (primeros 10 tokens): {sentences_poesia[0][:10]}')
print(f'Ejemplo rap   (primeros 10 tokens): {sentences_rap[0][:10]}')

### 2.2 Preprocesado con spaCy

Aplicamos las siguientes tareas de preprocesado para maximizar el rendimiento:
- **Minúsculas**: reducción de vocabulario y normalización
- **Eliminación de puntuación**: reducción de ruido
- **Eliminación de números**: poco informativos para distinción poesía/rap
- **Lematización**: reducción a forma canónica con spaCy (mejor que stemming para español)
- **Eliminación de stopwords**: con lista personalizada más la de spaCy

**Los tokens especiales `<EOL>` se preservan** ya que son informativos  
(indican la estructura métrica/rítmica, diferente entre poesía y rap).

**Este mismo preprocesado se aplica a cualquier corpus de test.**

In [ ]:
# ── PREPROCESADO ──────────────────────────────────────────────────────────────

def preprocess_sentence(tokens, nlp_model, remove_stopwords=True):
    """
    Preprocesa una lista de tokens (una canción/poema).

    Pasos:
      1. Preserva <EOL> (token especial informativo)
      2. Minúsculas
      3. Elimina puntuación y números
      4. Lematización con spaCy
      5. Eliminación de stopwords (opcional)

    IMPORTANTE: Aplicar exactamente este mismo preprocesado al corpus del profesor.

    Args:
        tokens          : Lista de tokens (salida de read_and_tokenize).
        nlp_model       : Modelo spaCy cargado.
        remove_stopwords: Si True, elimina stopwords.

    Returns:
        Lista de tokens preprocesados.
    """
    processed = []
    # Separamos los tokens especiales de los normales
    text_tokens = []
    positions   = []  # (posición_en_output, token_especial)

    for i, tok in enumerate(tokens):
        if tok == '<EOL>':
            positions.append((len(text_tokens), '<EOL>'))
        elif tok.startswith('<') and tok.endswith('>'):
            # Otro token especial: ignorar
            pass
        else:
            # Token normal: minúsculas
            text_tokens.append(tok.lower())

    # Construimos texto para spaCy (unimos con espacio)
    text = ' '.join(text_tokens)
    doc  = nlp_model(text)

    # Procesamos con spaCy: lematización + stopwords + puntuación + números
    lemmas = []
    for token in doc:
        if token.is_punct:
            continue
        if token.like_num or token.is_digit:
            continue
        if remove_stopwords and token.is_stop:
            continue
        if not token.text.strip():
            continue
        lemma = token.lemma_.lower().strip()
        if lemma:
            lemmas.append(lemma)

    # Reinsertamos <EOL> en las posiciones relativas correctas
    # Insertamos <EOL> cada vez que había uno en la secuencia original,
    # distribuyéndolos proporcionalmente.
    n_orig  = len(text_tokens)
    n_lemma = len(lemmas)

    if n_orig == 0 or n_lemma == 0:
        # Si no hay nada, devolvemos solo los EOL
        return ['<EOL>'] * len(positions)

    result = list(lemmas)  # copia
    # Insertamos <EOL> al final de cada segmento (posición relativa escalada)
    eol_positions_scaled = []
    for pos, _ in positions:
        scaled = int(pos * n_lemma / n_orig)
        scaled = min(scaled, n_lemma)
        eol_positions_scaled.append(scaled)

    # Insertamos de atrás hacia adelante para no desplazar índices
    for pos in sorted(set(eol_positions_scaled), reverse=True):
        result.insert(pos, '<EOL>')

    return result


def preprocess_corpus(sentences, nlp_model, remove_stopwords=True):
    """
    Aplica el preprocesado a todas las canciones de un corpus.
    Elimina canciones que queden vacías tras el preprocesado.
    """
    processed = []
    for sent in sentences:
        p = preprocess_sentence(sent, nlp_model, remove_stopwords)
        if len(p) > 1:  # Al menos 2 tokens
            processed.append(p)
    return processed


print('Preprocesando corpus de poesía...')
sentences_poesia_prep = preprocess_corpus(sentences_poesia, nlp)
print(f'  Antes: {len(sentences_poesia)} textos  →  Después: {len(sentences_poesia_prep)} textos')

print('Preprocesando corpus de rap...')
sentences_rap_prep = preprocess_corpus(sentences_rap, nlp)
print(f'  Antes: {len(sentences_rap)} textos  →  Después: {len(sentences_rap_prep)} textos')

print(f'\nEjemplo poesía original  : {sentences_poesia[0][:12]}')
print(f'Ejemplo poesía procesado : {sentences_poesia_prep[0][:12]}')
print(f'\nEjemplo rap original     : {sentences_rap[0][:12]}')
print(f'Ejemplo rap procesado    : {sentences_rap_prep[0][:12]}')

### 2.3 Construcción del Vocabulario

*(Código de práctica A83/A84 — sin cambios)*

In [ ]:
# ── CÓDIGO DE PRÁCTICAS A83 y A84 (Vocabulary — idéntico en ambas) ────────────

from collections import Counter

class Vocabulary:
    """
    Diccionario bidireccional: palabra <-> índice numérico.

    Tokens especiales reservados:
        <PAD>  índice 0 — relleno para longitud fija
        <UNK>  índice 1 — tokens desconocidos o de baja frecuencia

    Atributos:
        token_to_idx (dict): 'vida' -> 5
        idx_to_token (dict):  5 -> 'vida'
        size (int): número total de tokens registrados
    """

    def __init__(self):
        self.token_to_idx = {}
        self.idx_to_token = {}
        self.size = 0

    def add_token(self, token):
        """Añade un token al vocabulario si no existe ya."""
        if token not in self.token_to_idx:
            self.token_to_idx[token] = self.size
            self.idx_to_token[self.size] = token
            self.size += 1

    def build_from_sentences(self, sentences, min_freq):
        """
        Construye el vocabulario a partir de una lista de frases,
        incluyendo solo los tokens cuya frecuencia sea >= min_freq.
        Los tokens con frecuencia < min_freq quedarán mapeados a <UNK>
        al codificar.

        Args:
            sentences (list[list[str]]): corpus tokenizado.
            min_freq  (int): frecuencia mínima para entrar en el vocabulario.
        """
        # Contamos frecuencias de todos los tokens
        freq = Counter(tok for sentence in sentences for tok in sentence)

        # Solo añadimos tokens que superen el umbral
        for token, count in freq.items():
            if count >= min_freq:
                self.add_token(token)

        discarded = sum(1 for c in freq.values() if c < min_freq)
        print(f'  Tokens únicos en corpus   : {len(freq)}')
        print(f'  Tokens descartados (< {min_freq}) : {discarded}  -> mapeados a <UNK>')
        print(f'  Tokens en vocabulario      : {self.size}')

    def encode(self, tokens):
        """
        Convierte lista de palabras en lista de índices numéricos.
        Los tokens fuera del vocabulario (desconocidos o de baja frecuencia)
        se mapean al índice de <UNK> (1).

        Ejemplo: ['vida', 'de', 'xyz'] -> [5, 8, 1]  # 'xyz' no está -> <UNK>
        """
        unk_idx = self.token_to_idx['<UNK>']
        return [self.token_to_idx.get(tok, unk_idx) for tok in tokens]

    def decode(self, indices):
        """
        Convierte lista de índices en lista de palabras.
        Ejemplo: [5, 8, 11] -> ['vida', 'de', 'mi']
        """
        return [self.idx_to_token[idx] for idx in indices]


print('Clase Vocabulary definida (código de prácticas A83/A84).')

### 2.4 División Train / Test y Construcción del Vocabulario

In [ ]:
# ── PARÁMETROS GLOBALES ───────────────────────────────────────────────────────
TRAIN_RATIO = 0.8    # 80% entrenamiento, 20% test
MIN_FREQ    = 2      # Frecuencia mínima para el vocabulario
RANDOM_SEED = 42

# Etiquetas: 0 = Poesía (P), 1 = Rap (R)
# NOTA: guardamos la etiqueta como string también para el CSV de salida
LABEL_POESIA = 0
LABEL_RAP    = 1
LABEL_STR    = {0: 'P', 1: 'R'}

# Combinamos y mezclamos
labels_poesia = [LABEL_POESIA] * len(sentences_poesia_prep)
labels_rap    = [LABEL_RAP]    * len(sentences_rap_prep)

combined = list(zip(
    sentences_poesia_prep + sentences_rap_prep,
    labels_poesia         + labels_rap
))
random.seed(RANDOM_SEED)
random.shuffle(combined)
sentences_all, labels_all = zip(*combined)

split_idx = int(len(sentences_all) * TRAIN_RATIO)
train_sentences = sentences_all[:split_idx]
test_sentences  = sentences_all[split_idx:]
train_labels    = labels_all[:split_idx]
test_labels     = labels_all[split_idx:]

print(f'División del corpus:')
print(f'  Total   : {len(sentences_all)} textos')
print(f'  Train   : {len(train_sentences)} textos  ({TRAIN_RATIO*100:.0f}%)')
print(f'  Test    : {len(test_sentences)} textos  ({(1-TRAIN_RATIO)*100:.0f}%)')
print(f'  Poesía  : {list(labels_all).count(0)} textos')
print(f'  Rap     : {list(labels_all).count(1)} textos')

# Construimos el vocabulario SOLO sobre entrenamiento (evitar data leakage)
vocab = Vocabulary()
vocab.add_token('<PAD>')   # índice 0 — relleno
vocab.add_token('<UNK>')   # índice 1 — desconocido / baja frecuencia

print(f'\nConstruyendo vocabulario con MIN_FREQ = {MIN_FREQ}...')
vocab.build_from_sentences(train_sentences, min_freq=MIN_FREQ)
print(f'\nVocabulario construido.')
print(f'  Tamaño total : {vocab.size} tokens')
print(f'  Índice <PAD> : {vocab.token_to_idx["<PAD>"]}')
print(f'  Índice <UNK> : {vocab.token_to_idx["<UNK>"]}')
print(f'  Índice <EOL> : {vocab.token_to_idx.get("<EOL>", "No encontrado")}')

### 2.5 Dataset de Clasificación (Padding/Truncado)

*(Código de práctica A83/A84 — `SentenceDataset` — sin cambios)*

In [ ]:
# ── CÓDIGO DE PRÁCTICAS A83 y A84 (SentenceDataset — idéntico en ambas) ───────

class SentenceDataset(Dataset):
    """
    Envuelve pares (secuencia_de_tokens_codificada, etiqueta) en el formato
    que PyTorch necesita para usar DataLoader.

    Aplica padding/truncado para garantizar longitud fija = context_size.
    Los tokens no presentes en el vocabulario se mapean a <UNK> durante
    la codificación (gestionado por vocab.encode).
    """

    def __init__(self, sentences, labels, vocab, context_size):
        self.vocab        = vocab
        self.context_size = context_size
        self.data         = []
        self.pad_idx      = vocab.token_to_idx['<PAD>']

        for sentence, label in zip(sentences, labels):
            # vocab.encode mapea tokens OOV o de baja frecuencia a <UNK>
            encoded = vocab.encode(sentence)

            # Padding o truncado hasta context_size
            if len(encoded) < context_size:
                encoded = encoded + [self.pad_idx] * (context_size - len(encoded))
            else:
                encoded = encoded[:context_size]

            self.data.append((encoded, label))

    def __len__(self):
        """Número total de ejemplos."""
        return len(self.data)

    def __getitem__(self, idx):
        """
        Devuelve el ejemplo número 'idx' como tensores PyTorch.

        Returns:
            inputs: Tensor Long [context_size]
            label:  Tensor Float [] (escalar 0 ó 1)
        """
        inputs, label = self.data[idx]
        return (
            torch.tensor(inputs, dtype=torch.long),
            torch.tensor(label,  dtype=torch.float)   # float para BCEWithLogitsLoss
        )


# ── Parámetros del dataset ────────────────────────────────────────────────────
# context_size más grande captura más contexto de cada canción
# Ajusta según el corpus (longitud media de los textos)
CONTEXT_SIZE = 200
BATCH_SIZE   = 32

train_dataset = SentenceDataset(train_sentences, train_labels, vocab, CONTEXT_SIZE)
test_dataset  = SentenceDataset(test_sentences,  test_labels,  vocab, CONTEXT_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Datasets y DataLoaders preparados.')
print(f'  context_size : {CONTEXT_SIZE} tokens por texto')
print(f'  batch_size   : {BATCH_SIZE}')
print(f'  Train — {len(train_dataset)} ejemplos  |  {len(train_loader)} batches')
print(f'  Test  — {len(test_dataset)} ejemplos   |  {len(test_loader)} batches')

# Inspeccionamos un ejemplo
inputs_ex, label_ex = train_dataset[0]
print(f'\nEjemplo:')
print(f'  Forma tensor entrada : {inputs_ex.shape}')
print(f'  Etiqueta             : {LABEL_STR[int(label_ex.item())]}')
print(f'  Primeras 8 palabras  : {vocab.decode(inputs_ex[:8].tolist())}')

In [ ]:
# ── DATASET ESPECÍFICO PARA RNN — con stopwords ───────────────────────────────
# La RNN procesa la secuencia ordenada token a token: las stopwords preservan
# la continuidad sintáctica y mejoran la clasificación.
# PAD (suma de embeddings) y Transformer (atención global) no se benefician
# igual, por eso solo la RNN usa este preprocesado alternativo.
print('Preprocesando corpus para RNN (con stopwords)...')
sentences_poesia_rnn = preprocess_corpus(sentences_poesia, nlp, remove_stopwords=False)
sentences_rap_rnn    = preprocess_corpus(sentences_rap,    nlp, remove_stopwords=False)

combined_rnn = list(zip(
    sentences_poesia_rnn + sentences_rap_rnn,
    [LABEL_POESIA] * len(sentences_poesia_rnn) + [LABEL_RAP] * len(sentences_rap_rnn)
))
random.seed(RANDOM_SEED)
random.shuffle(combined_rnn)
sentences_rnn_all, labels_rnn_all = zip(*combined_rnn)

split_rnn           = int(len(sentences_rnn_all) * TRAIN_RATIO)
train_sentences_rnn = sentences_rnn_all[:split_rnn]
test_sentences_rnn  = sentences_rnn_all[split_rnn:]
train_labels_rnn    = labels_rnn_all[:split_rnn]
test_labels_rnn     = labels_rnn_all[split_rnn:]

# Vocabulario propio para la RNN (con stopwords → más tokens → más contexto)
vocab_rnn = Vocabulary()
vocab_rnn.add_token('<PAD>')
vocab_rnn.add_token('<UNK>')
print(f'Construyendo vocabulario RNN con MIN_FREQ = {MIN_FREQ}...')
vocab_rnn.build_from_sentences(train_sentences_rnn, min_freq=MIN_FREQ)

# context_size mayor: los textos conservan stopwords y son más largos
CONTEXT_SIZE_RNN = 300
BATCH_SIZE_RNN   = 32

train_dataset_rnn = SentenceDataset(train_sentences_rnn, train_labels_rnn, vocab_rnn, CONTEXT_SIZE_RNN)
test_dataset_rnn  = SentenceDataset(test_sentences_rnn,  test_labels_rnn,  vocab_rnn, CONTEXT_SIZE_RNN)
train_loader_rnn  = DataLoader(train_dataset_rnn, batch_size=BATCH_SIZE_RNN, shuffle=True)
test_loader_rnn   = DataLoader(test_dataset_rnn,  batch_size=BATCH_SIZE_RNN, shuffle=False)

print(f'  Train RNN : {len(train_dataset_rnn)} ejemplos  |  Test RNN : {len(test_dataset_rnn)} ejemplos')
print(f'  Vocab RNN : {vocab_rnn.size} tokens  |  context_size_rnn : {CONTEXT_SIZE_RNN}')

---
## PARTE 3: Modelos Neuronales

### 3.1 Funciones de Entrenamiento y Evaluación Compartidas

*(Código de prácticas A83/A84 — `train_model` y `evaluate_model` — sin cambios)*

In [ ]:
# ── CÓDIGO DE PRÁCTICAS A83 y A84 (train_model — idéntico en ambas) ───────────

def train_model(model, data_loader, num_epochs, learning_rate, device='cpu'):
    """
    Entrena el clasificador binario.

    Args:
        model         : Modelo (BengioNN o ElmanRNN).
        data_loader   : DataLoader con los datos de entrenamiento.
        num_epochs    : Número de vueltas completas al dataset.
        learning_rate : Tasa de aprendizaje para Adam.
        device        : 'cpu' o 'cuda'.
    """
    model.to(device)
    model.train()   # Modo entrenamiento

    # BCEWithLogitsLoss: adecuada para clasificación binaria
    criterion = nn.BCEWithLogitsLoss()

    # Adam: ajusta la tasa de aprendizaje automáticamente para cada parámetro
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    total_batches = len(data_loader)
    print(f'Iniciando entrenamiento en {device}')
    print(f'  Épocas: {num_epochs}  |  LR: {learning_rate}  |  Batch: {data_loader.batch_size}')
    print('-' * 65)

    for epoch in range(num_epochs):
        total_loss = 0.0

        for batch_idx, (inputs, labels) in enumerate(data_loader):
            # Movemos los datos al mismo dispositivo que el modelo
            inputs = inputs.to(device)           # [batch, context_size]
            labels = labels.to(device).float()   # [batch]  — float para BCEWithLogitsLoss

            # ── PASO 1: Forward pass ──────────────────────────────────────
            outputs = model(inputs)              # [batch]

            # ── PASO 2: Cálculo de la pérdida ────────────────────────────
            loss = criterion(outputs, labels)

            # ── PASO 3: Backward pass ────────────────────────────────────
            optimizer.zero_grad()   # Limpiamos gradientes del paso anterior
            loss.backward()         # Calculamos gradientes

            # ── PASO 4: Actualización de pesos ───────────────────────────
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / total_batches
        print(f'Época [{epoch+1:3d}/{num_epochs}]  |  Loss: {avg_loss:.4f}')

    print('-' * 65)
    print('Entrenamiento completado.')


# ── CÓDIGO DE PRÁCTICAS A83 y A84 (evaluate_model — idéntico en ambas) ────────

def evaluate_model(model, data_loader, device='cpu'):
    """
    Evalúa la exactitud del clasificador en un conjunto de datos.

    Args:
        model       : Modelo ya entrenado.
        data_loader : DataLoader con los datos de evaluación.
        device      : 'cpu' o 'cuda'.

    Returns:
        float: Exactitud (accuracy) entre 0 y 1.
    """
    model.to(device)
    model.eval()    # Modo evaluación: desactiva comportamientos de entrenamiento

    correct = 0
    total   = 0

    with torch.no_grad():   # No calculamos gradientes: ahorra memoria y tiempo
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)   # [batch]  — logits

            # Convertimos logits a predicciones de clase (0 ó 1)
            predictions = (torch.sigmoid(outputs) > 0.5).long()
            labels_long = labels.long()

            correct += (predictions == labels_long).sum().item()
            total   += labels.size(0)

    return correct / total


print('Funciones train_model y evaluate_model definidas (código de prácticas A83/A84).')

---
### 3.2 Modelo PAD — Red de Bengio Clasificadora

*(Código **estrictamente** de práctica A83 — `BengioNN` — sin cambios)*

**Arquitectura:** Embedding → Suma → Capa oculta (ReLU) → 1 logit (clasificación binaria)

In [ ]:
# ── CÓDIGO DE PRÁCTICA A83 (BengioNN clasificador — sin cambios) ──────────────

class BengioNN(nn.Module):
    """
    Red Neuronal de Bengio adaptada a clasificación binaria.

    Dado un conjunto de tokens (secuencia de una canción), predice
    a qué clase pertenece (0 ó 1).

    Arquitectura:
        Embedding -> Suma sobre tokens -> Capa oculta (ReLU) -> 1 logit

    Args:
        vocab_size   (int): Número de palabras en el vocabulario.
        context_size (int): Número de tokens de entrada (longitud de secuencia).
        embed_size   (int): Dimensión de cada vector de embedding.
        hidden_dim   (int): Número de neuronas en la capa oculta.
    """

    def __init__(self, vocab_size, context_size, embed_size, hidden_dim):
        super(BengioNN, self).__init__()

        self.context_size = context_size

        # Tabla de embeddings: vocab_size filas, cada una de embed_size dimensiones
        # Se inicializa aleatoriamente y se APRENDE durante el entrenamiento
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Capa oculta: recibe la suma de embeddings
        # Entrada: embed_size   Salida: hidden_dim
        self.fc1 = nn.Linear(embed_size, hidden_dim)

        # Capa de salida: un único logit para clasificación binaria
        # Entrada: hidden_dim   Salida: 1
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        """
        Propagación hacia adelante (forward pass).

        Args:
            x: Tensor [batch_size, context_size] con índices de palabras.

        Returns:
            logits: Tensor [batch_size] (un valor por ejemplo).
        """
        # 1. Lookup de embeddings
        #    [batch, context_size] -> [batch, context_size, embed_size]
        emb = self.embedding(x)

        # 2. SUMA sobre la dimensión de los tokens
        #    [batch, context_size, embed_size] -> [batch, embed_size]
        #    Cada token contribuye con su vector; la suma agrega la información
        summed = torch.sum(emb, dim=1)

        # 3. Capa oculta con ReLU (no-linealidad)
        #    [batch, embed_size] -> [batch, hidden_dim]
        hidden = F.relu(self.fc1(summed))

        # 4. Capa de salida: un logit por ejemplo
        #    [batch, hidden_dim] -> [batch, 1] -> [batch]
        logits = self.fc2(hidden).squeeze(-1)

        return logits

#### 3.2.1 Búsqueda de Hiperparámetros — Modelo PAD

Evaluamos combinaciones de hiperparámetros para encontrar los óptimos.

In [ ]:
# ── BÚSQUEDA DE HIPERPARÁMETROS — Bengio PAD ─────────────────────────────────
# Evaluamos diferentes configuraciones y seleccionamos la mejor

param_grid_pad = [
    {'embed_size': 64,  'hidden_dim': 128, 'lr': 0.001,  'epochs': 20},
    {'embed_size': 128, 'hidden_dim': 256, 'lr': 0.001,  'epochs': 20},
    {'embed_size': 128, 'hidden_dim': 256, 'lr': 0.0001, 'epochs': 30},
    {'embed_size': 256, 'hidden_dim': 512, 'lr': 0.001,  'epochs': 20},
]

best_acc_pad  = 0.0
best_cfg_pad  = None

print('== Búsqueda de hiperparámetros para modelo PAD (Bengio) ==')
for cfg in param_grid_pad:
    print(f'\nConfig: {cfg}')
    model_pad_tmp = BengioNN(
        vocab_size   = vocab.size,
        context_size = CONTEXT_SIZE,
        embed_size   = cfg['embed_size'],
        hidden_dim   = cfg['hidden_dim']
    )
    train_model(model_pad_tmp, train_loader, cfg['epochs'], cfg['lr'], device=device)
    acc = evaluate_model(model_pad_tmp, test_loader, device=device)
    print(f'  → Test accuracy: {acc:.4f} ({acc*100:.1f}%)')
    if acc > best_acc_pad:
        best_acc_pad = acc
        best_cfg_pad = cfg
        torch.save(model_pad_tmp.state_dict(), f'{OUTPUT_DIR}/pad_best.pth')
        best_pad_params = {
            'vocab_size': vocab.size,
            'context_size': CONTEXT_SIZE,
            'embed_size': cfg['embed_size'],
            'hidden_dim': cfg['hidden_dim']
        }

print(f'\n== Mejor configuración PAD ==')
print(f'  Config   : {best_cfg_pad}')
print(f'  Accuracy : {best_acc_pad:.4f} ({best_acc_pad*100:.1f}%)')

#### 3.2.2 Entrenamiento Final — Modelo PAD

In [ ]:
# ── ENTRENAMIENTO FINAL CON LA MEJOR CONFIGURACIÓN ───────────────────────────

print(f'Entrenando modelo PAD con configuración óptima: {best_cfg_pad}')

model_pad = BengioNN(
    vocab_size   = vocab.size,
    context_size = CONTEXT_SIZE,
    embed_size   = best_cfg_pad['embed_size'],
    hidden_dim   = best_cfg_pad['hidden_dim']
)

total_params_pad = sum(p.numel() for p in model_pad.parameters() if p.requires_grad)
print(f'\nArquitectura PAD:')
print(model_pad)
print(f'\nParámetros entrenables: {total_params_pad:,}')

train_model(model_pad, train_loader, best_cfg_pad['epochs'], best_cfg_pad['lr'], device=device)

train_acc_pad = evaluate_model(model_pad, train_loader, device=device)
test_acc_pad  = evaluate_model(model_pad, test_loader,  device=device)

print(f'\nResultados Modelo PAD (Bengio):')
print(f'  Train accuracy : {train_acc_pad:.4f} ({train_acc_pad*100:.1f}%)')
print(f'  Test accuracy  : {test_acc_pad:.4f}  ({test_acc_pad*100:.1f}%)')

# Guardamos el modelo entrenado
torch.save(model_pad.state_dict(), f'{OUTPUT_DIR}/modelo_pad_final.pth')
print(f'\nModelo PAD guardado en {OUTPUT_DIR}/modelo_pad_final.pth')

---
### 3.3 Modelo RNN — Red Recurrente de Elman Clasificadora

*(Código **estrictamente** de práctica A84 — `ElmanRNN` — sin cambios)*

**Arquitectura:** Embedding → RNN → Estado oculto final h_T → fc → 1 logit

In [ ]:
# ── CÓDIGO DE PRÁCTICA A84 (ElmanRNN clasificador — sin cambios) ──────────────

class ElmanRNN(nn.Module):
    """
    Red Neuronal Recurrente de Elman adaptada a clasificación binaria.

    Procesa la secuencia de tokens de una canción y predice su clase (0 ó 1)
    usando el estado oculto del último token como representación global.

    Arquitectura:
        Embedding -> nn.RNN -> fc (1 logit) usando hidden[-1]

    Args:
        vocab_size (int):  Número de palabras en el vocabulario.
        embed_size (int):  Dimensión de los vectores de embedding.
        hidden_dim (int):  Dimensión del estado oculto de la RNN.
    """

    def __init__(self, vocab_size, embed_size, hidden_dim):
        super(ElmanRNN, self).__init__()

        # Tabla de embeddings — idéntica a todas las prácticas anteriores
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Capa RNN de Elman
        # batch_first=True -> espera [batch, seq_len, embed_size]
        self.rnn = nn.RNN(embed_size, hidden_dim, batch_first=True)

        # Capa de salida: UN único logit para clasificación binaria
        # Recibe el estado oculto final h_T (tamaño hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        """
        Propagación hacia adelante.

        Args:
            x : Tensor [batch, seq_len] — índices de tokens.

        Returns:
            logits : Tensor [batch] — un logit por ejemplo.
        """
        # 1. Lookup de embeddings
        #    [batch, seq_len] -> [batch, seq_len, embed_size]
        emb = self.embedding(x)

        # 2. Capa RNN — procesa toda la secuencia
        #    output : [batch, seq_len, hidden_dim]  — todos los estados ocultos
        #    hidden : [1, batch, hidden_dim]         — estado oculto final h_T
        output, hidden = self.rnn(emb)

        # 3. Usamos SOLO el estado oculto final como representación de la canción
        #    hidden[-1] : [batch, hidden_dim]
        #    (para RNN de 1 capa: hidden[-1] == output[:, -1, :])
        final_hidden = hidden[-1]   # [batch, hidden_dim]

        # 4. Capa de salida: 1 logit por ejemplo
        #    [batch, hidden_dim] -> [batch, 1] -> [batch]
        logits = self.fc(final_hidden).squeeze(-1)

        return logits

#### 3.3.1 Búsqueda de Hiperparámetros — Modelo RNN

In [ ]:
# ── BÚSQUEDA DE HIPERPARÁMETROS — ElmanRNN ───────────────────────────────────
# Usa train_loader_rnn / test_loader_rnn (corpus con stopwords, context=300)
# y vocab_rnn.size (vocabulario más grande al conservar stopwords).

param_grid_rnn = [
    {'embed_size': 128, 'hidden_dim': 256, 'lr': 0.001,  'epochs': 30},
    {'embed_size': 128, 'hidden_dim': 512, 'lr': 0.001,  'epochs': 30},
    {'embed_size': 256, 'hidden_dim': 512, 'lr': 0.001,  'epochs': 30},
    {'embed_size': 256, 'hidden_dim': 512, 'lr': 0.0005, 'epochs': 40},
]

best_acc_rnn = 0.0
best_cfg_rnn = None

print('== Búsqueda de hiperparámetros para modelo RNN (Elman) ==')
for cfg in param_grid_rnn:
    print(f'\nConfig: {cfg}')
    model_rnn_tmp = ElmanRNN(
        vocab_size = vocab_rnn.size,
        embed_size = cfg['embed_size'],
        hidden_dim = cfg['hidden_dim']
    )
    train_model(model_rnn_tmp, train_loader_rnn, cfg['epochs'], cfg['lr'], device=device)
    acc = evaluate_model(model_rnn_tmp, test_loader_rnn, device=device)
    print(f'  → Test accuracy: {acc:.4f} ({acc*100:.1f}%)')
    if acc > best_acc_rnn:
        best_acc_rnn = acc
        best_cfg_rnn = cfg
        torch.save(model_rnn_tmp.state_dict(), f'{OUTPUT_DIR}/rnn_best.pth')
        best_rnn_params = {
            'vocab_size': vocab_rnn.size,
            'embed_size': cfg['embed_size'],
            'hidden_dim': cfg['hidden_dim']
        }

print(f'\n== Mejor configuración RNN ==')
print(f'  Config   : {best_cfg_rnn}')
print(f'  Accuracy : {best_acc_rnn:.4f} ({best_acc_rnn*100:.1f}%)')

#### 3.3.2 Entrenamiento Final — Modelo RNN

In [ ]:
# ── ENTRENAMIENTO FINAL CON LA MEJOR CONFIGURACIÓN ───────────────────────────

print(f'Entrenando modelo RNN con configuración óptima: {best_cfg_rnn}')

model_rnn = ElmanRNN(
    vocab_size = vocab_rnn.size,
    embed_size = best_cfg_rnn['embed_size'],
    hidden_dim = best_cfg_rnn['hidden_dim']
)

total_params_rnn = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)
print(f'\nArquitectura RNN:')
print(model_rnn)
print(f'\nParámetros entrenables: {total_params_rnn:,}')

train_model(model_rnn, train_loader_rnn, best_cfg_rnn['epochs'], best_cfg_rnn['lr'], device=device)

train_acc_rnn = evaluate_model(model_rnn, train_loader_rnn, device=device)
test_acc_rnn  = evaluate_model(model_rnn, test_loader_rnn,  device=device)

print(f'\nResultados Modelo RNN (Elman):')
print(f'  Train accuracy : {train_acc_rnn:.4f} ({train_acc_rnn*100:.1f}%)')
print(f'  Test accuracy  : {test_acc_rnn:.4f}  ({test_acc_rnn*100:.1f}%)')

torch.save(model_rnn.state_dict(), f'{OUTPUT_DIR}/modelo_rnn_final.pth')
print(f'\nModelo RNN guardado en {OUTPUT_DIR}/modelo_rnn_final.pth')

---
### 3.4 Modelo Transformer

Implementado con **las mismas primitivas de PyTorch** (`torch.nn`) usadas en las prácticas,  
sin librerías externas adicionales. Utiliza la arquitectura Transformer estándar con:
- **Positional Encoding** sinusoidal (añadido al embedding)
- **Encoder** con Multi-Head Self-Attention y Feed-Forward
- **Clasificación** a partir del token `[CLS]` (primer token de la secuencia)

In [ ]:
# ── MODELO TRANSFORMER — implementado con torch.nn (mismo framework que prácticas)

class PositionalEncoding(nn.Module):
    """
    Añade información posicional a los embeddings.
    Usa la codificación sinusoidal estándar (Vaswani et al., 2017).

    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Calculamos la tabla de positional encodings de forma estática
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # [max_len, 1]
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)   # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: Tensor [batch, seq_len, d_model]
        Returns:
            Tensor [batch, seq_len, d_model] con positional encoding sumado
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerClassifier(nn.Module):
    """
    Clasificador binario basado en Transformer Encoder.

    Arquitectura:
        Embedding → PositionalEncoding → TransformerEncoder → Pooling → fc → 1 logit

    Usa nn.TransformerEncoder (igual que nn.RNN en las prácticas de Elman).
    La clasificación se hace sobre la media de todos los estados de salida
    del encoder (mean pooling), más robusto que usar solo el primer token.

    Args:
        vocab_size  (int): Tamaño del vocabulario.
        embed_size  (int): Dimensión del embedding (= d_model del Transformer).
        num_heads   (int): Número de cabezas de atención (debe dividir embed_size).
        hidden_dim  (int): Dimensión de la capa feed-forward interna.
        num_layers  (int): Número de capas del encoder.
        max_len     (int): Longitud máxima de secuencia para positional encoding.
        dropout     (float): Tasa de dropout.
    """

    def __init__(self, vocab_size, embed_size, num_heads, hidden_dim,
                 num_layers, max_len=512, dropout=0.1):
        super(TransformerClassifier, self).__init__()

        # Tabla de embeddings — igual que en BengioNN y ElmanRNN
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)

        # Positional encoding sinusoidal
        self.pos_encoding = PositionalEncoding(embed_size, max_len=max_len, dropout=dropout)

        # Encoder del Transformer (usa nn.TransformerEncoderLayer, igual que
        # nn.RNN es la capa recurrente en ElmanRNN)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = embed_size,
            nhead           = num_heads,
            dim_feedforward = hidden_dim,
            dropout         = dropout,
            batch_first     = True     # [batch, seq, features]
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers = num_layers
        )

        # Capa de salida: 1 logit para clasificación binaria
        # Igual que fc en ElmanRNN clasificador
        self.fc = nn.Linear(embed_size, 1)

    def forward(self, x):
        """
        Propagación hacia adelante.

        Args:
            x : Tensor [batch, seq_len] — índices de tokens.

        Returns:
            logits : Tensor [batch] — un logit por ejemplo.
        """
        # Máscara de padding: True donde hay <PAD> (índice 0)
        # El encoder ignorará esas posiciones en la atención
        padding_mask = (x == 0)   # [batch, seq_len]

        # 1. Embedding + positional encoding
        #    [batch, seq_len] -> [batch, seq_len, embed_size]
        emb = self.embedding(x)
        emb = self.pos_encoding(emb)

        # 2. Transformer encoder
        #    [batch, seq_len, embed_size] -> [batch, seq_len, embed_size]
        encoded = self.transformer_encoder(emb, src_key_padding_mask=padding_mask)

        # 3. Mean pooling sobre tokens no-PAD
        #    Promediamos solo los tokens que no son padding
        mask_expanded = (~padding_mask).unsqueeze(-1).float()   # [batch, seq_len, 1]
        sum_encoded   = (encoded * mask_expanded).sum(dim=1)    # [batch, embed_size]
        count         = mask_expanded.sum(dim=1).clamp(min=1)   # [batch, 1]
        pooled        = sum_encoded / count                      # [batch, embed_size]

        # 4. Capa de salida: 1 logit por ejemplo
        #    [batch, embed_size] -> [batch, 1] -> [batch]
        logits = self.fc(pooled).squeeze(-1)

        return logits


print('Clase TransformerClassifier definida (torch.nn, mismas primitivas que prácticas).')

#### 3.4.1 Búsqueda de Hiperparámetros — Modelo Transformer

In [ ]:
# ── BÚSQUEDA DE HIPERPARÁMETROS — Transformer ────────────────────────────────

param_grid_tf = [
    {'embed_size': 64,  'num_heads': 4, 'hidden_dim': 128, 'num_layers': 2, 'lr': 0.001,  'epochs': 20},
    {'embed_size': 128, 'num_heads': 4, 'hidden_dim': 256, 'num_layers': 2, 'lr': 0.001,  'epochs': 20},
    {'embed_size': 128, 'num_heads': 8, 'hidden_dim': 256, 'num_layers': 3, 'lr': 0.0001, 'epochs': 30},
    {'embed_size': 256, 'num_heads': 8, 'hidden_dim': 512, 'num_layers': 3, 'lr': 0.0001, 'epochs': 30},
]

best_acc_tf = 0.0
best_cfg_tf = None

print('== Búsqueda de hiperparámetros para modelo Transformer ==')
for cfg in param_grid_tf:
    print(f'\nConfig: {cfg}')
    model_tf_tmp = TransformerClassifier(
        vocab_size = vocab.size,
        embed_size = cfg['embed_size'],
        num_heads  = cfg['num_heads'],
        hidden_dim = cfg['hidden_dim'],
        num_layers = cfg['num_layers'],
        max_len    = CONTEXT_SIZE + 10
    )
    train_model(model_tf_tmp, train_loader, cfg['epochs'], cfg['lr'], device=device)
    acc = evaluate_model(model_tf_tmp, test_loader, device=device)
    print(f'  → Test accuracy: {acc:.4f} ({acc*100:.1f}%)')
    if acc > best_acc_tf:
        best_acc_tf = acc
        best_cfg_tf = cfg
        torch.save(model_tf_tmp.state_dict(), f'{OUTPUT_DIR}/tf_best.pth')
        best_tf_params = {
            'vocab_size': vocab.size,
            'embed_size': cfg['embed_size'],
            'num_heads' : cfg['num_heads'],
            'hidden_dim': cfg['hidden_dim'],
            'num_layers': cfg['num_layers'],
            'max_len'   : CONTEXT_SIZE + 10
        }

print(f'\n== Mejor configuración Transformer ==')
print(f'  Config   : {best_cfg_tf}')
print(f'  Accuracy : {best_acc_tf:.4f} ({best_acc_tf*100:.1f}%)')

#### 3.4.2 Entrenamiento Final — Modelo Transformer

In [ ]:
# ── ENTRENAMIENTO FINAL CON LA MEJOR CONFIGURACIÓN ───────────────────────────

print(f'Entrenando modelo Transformer con configuración óptima: {best_cfg_tf}')

model_tf = TransformerClassifier(
    vocab_size = vocab.size,
    embed_size = best_cfg_tf['embed_size'],
    num_heads  = best_cfg_tf['num_heads'],
    hidden_dim = best_cfg_tf['hidden_dim'],
    num_layers = best_cfg_tf['num_layers'],
    max_len    = CONTEXT_SIZE + 10
)

total_params_tf = sum(p.numel() for p in model_tf.parameters() if p.requires_grad)
print(f'\nArquitectura Transformer:')
print(model_tf)
print(f'\nParámetros entrenables: {total_params_tf:,}')

train_model(model_tf, train_loader, best_cfg_tf['epochs'], best_cfg_tf['lr'], device=device)

train_acc_tf = evaluate_model(model_tf, train_loader, device=device)
test_acc_tf  = evaluate_model(model_tf, test_loader,  device=device)

print(f'\nResultados Modelo Transformer:')
print(f'  Train accuracy : {train_acc_tf:.4f} ({train_acc_tf*100:.1f}%)')
print(f'  Test accuracy  : {test_acc_tf:.4f}  ({test_acc_tf*100:.1f}%)')

torch.save(model_tf.state_dict(), f'{OUTPUT_DIR}/modelo_transformer_final.pth')
print(f'\nModelo Transformer guardado en {OUTPUT_DIR}/modelo_transformer_final.pth')

---
### 3.5 Resumen Comparativo de Modelos

In [ ]:
print('=' * 65)
print('RESUMEN DE RESULTADOS')
print('=' * 65)
print(f'{"Modelo":<20} {"Train Acc":>12} {"Test Acc":>12}')
print('-' * 65)
print(f'{"PAD (Bengio)":<20} {train_acc_pad:>11.1%} {test_acc_pad:>11.1%}')
print(f'{"RNN (Elman)":<20} {train_acc_rnn:>11.1%} {test_acc_rnn:>11.1%}')
print(f'{"Transformer":<20} {train_acc_tf:>11.1%} {test_acc_tf:>11.1%}')
print('=' * 65)

# Estimación del error
print('\nEstimación del error (sobre el 20% de test):')
print(f'  PAD         : {(1-test_acc_pad)*100:.1f}%')
print(f'  RNN         : {(1-test_acc_rnn)*100:.1f}%')
print(f'  Transformer : {(1-test_acc_tf)*100:.1f}%')

---
## PARTE 4: Clasificación del Corpus del Profesor

### 4.1 Función de Clasificación

Esta función recibe un corpus en el formato `<CS>...</CS>` / `<EOL>` y devuelve  
las predicciones de los tres modelos, aplicando **exactamente el mismo preprocesado**  
que se usó durante el entrenamiento.

In [ ]:
def classify_corpus(corpus_path, model_pad, model_rnn, model_tf,
                    vocab, vocab_rnn, context_size, context_size_rnn,
                    nlp_model, device='cpu'):
    """
    Clasifica todos los textos de un corpus con los tres modelos.

    Pasos:
      1. Lee y tokeniza el corpus (mismo read_and_tokenize de las prácticas)
      2. Preprocesado CON stopwords para la RNN, SIN stopwords para PAD y Transformer
      3. Codifica con el vocabulario correspondiente a cada modelo
      4. Clasifica con PAD, RNN y Transformer
      5. Devuelve lista de tuplas (num_texto, pred_pad, pred_rnn, pred_tf)

    Args:
        corpus_path      : Ruta al corpus en formato <CS>...</CS>.
        model_pad        : Modelo BengioNN entrenado.
        model_rnn        : Modelo ElmanRNN entrenado.
        model_tf         : Modelo TransformerClassifier entrenado.
        vocab            : Vocabulario de PAD y Transformer (sin stopwords).
        vocab_rnn        : Vocabulario de la RNN (con stopwords).
        context_size     : Longitud de secuencia para PAD y Transformer.
        context_size_rnn : Longitud de secuencia para la RNN.
        nlp_model        : Modelo spaCy (mismo que en entrenamiento).
        device           : 'cpu' o 'cuda'.

    Returns:
        List[tuple]: [(1, 'P'/'R', 'P'/'R', 'P'/'R'), ...]
    """
    # ── Paso 1: Leer y tokenizar ──────────────────────────────────────────────
    raw_sentences = read_and_tokenize(corpus_path)
    print(f'  Textos leídos : {len(raw_sentences)}')

    # ── Paso 2a: Preprocesado sin stopwords (PAD + Transformer) ──────────────
    processed_pad_tf = [preprocess_sentence(s, nlp_model, remove_stopwords=True)
                        for s in raw_sentences]

    # ── Paso 2b: Preprocesado con stopwords (RNN) ─────────────────────────────
    processed_rnn = [preprocess_sentence(s, nlp_model, remove_stopwords=False)
                     for s in raw_sentences]

    # ── Paso 3a: Codificar para PAD y Transformer ─────────────────────────────
    pad_idx = vocab.token_to_idx['<PAD>']
    encoded_pad_tf = []
    for sent in processed_pad_tf:
        enc = vocab.encode(sent)
        if len(enc) < context_size:
            enc = enc + [pad_idx] * (context_size - len(enc))
        else:
            enc = enc[:context_size]
        encoded_pad_tf.append(enc)

    # ── Paso 3b: Codificar para RNN ───────────────────────────────────────────
    pad_idx_rnn = vocab_rnn.token_to_idx['<PAD>']
    encoded_rnn = []
    for sent in processed_rnn:
        enc = vocab_rnn.encode(sent)
        if len(enc) < context_size_rnn:
            enc = enc + [pad_idx_rnn] * (context_size_rnn - len(enc))
        else:
            enc = enc[:context_size_rnn]
        encoded_rnn.append(enc)

    inputs_pad_tf = torch.tensor(encoded_pad_tf, dtype=torch.long).to(device)
    inputs_rnn    = torch.tensor(encoded_rnn,    dtype=torch.long).to(device)

    # ── Paso 4: Clasificación (modo evaluación, sin gradientes) ──────────────
    for m in [model_pad, model_rnn, model_tf]:
        m.to(device)
        m.eval()

    preds_pad, preds_rnn, preds_tf = [], [], []
    BATCH = 64

    with torch.no_grad():
        for start in range(0, len(inputs_pad_tf), BATCH):
            batch_pad_tf = inputs_pad_tf[start:start+BATCH]
            batch_rnn    = inputs_rnn[start:start+BATCH]

            out_pad = (torch.sigmoid(model_pad(batch_pad_tf)) > 0.5).long().cpu().tolist()
            out_rnn = (torch.sigmoid(model_rnn(batch_rnn))    > 0.5).long().cpu().tolist()
            out_tf  = (torch.sigmoid(model_tf(batch_pad_tf))  > 0.5).long().cpu().tolist()

            preds_pad.extend(out_pad)
            preds_rnn.extend(out_rnn)
            preds_tf.extend(out_tf)

    # ── Paso 5: Formatear resultados ──────────────────────────────────────────
    results = []
    for i, (p, r, t) in enumerate(zip(preds_pad, preds_rnn, preds_tf), start=1):
        results.append((i, LABEL_STR[p], LABEL_STR[r], LABEL_STR[t]))

    return results


print('Función classify_corpus definida.')

### 4.2 Clasificación y Generación del CSV

**Actualiza la ruta al corpus del profesor y tu número de alumno.**

In [ ]:
# ── CONFIGURACIÓN ─────────────────────────────────────────────────────────────
# Actualiza con el corpus que proporcione el profesor
PATH_CORPUS_PROFESOR = './corpus_test_profesor.txt'

# Actualiza con tu número de alumno (alu + número sin espacios)
NUMERO_ALU = 'TUNUM'   # Ejemplo: '123456'

# Ruta del fichero CSV de salida
CSV_OUTPUT = f'./alu{NUMERO_ALU}.csv'

# ── CLASIFICACIÓN ─────────────────────────────────────────────────────────────
print(f'Clasificando corpus del profesor: {PATH_CORPUS_PROFESOR}')
resultados = classify_corpus(
    corpus_path      = PATH_CORPUS_PROFESOR,
    model_pad        = model_pad,
    model_rnn        = model_rnn,
    model_tf         = model_tf,
    vocab            = vocab,
    vocab_rnn        = vocab_rnn,
    context_size     = CONTEXT_SIZE,
    context_size_rnn = CONTEXT_SIZE_RNN,
    nlp_model        = nlp,
    device           = device
)

# ── ESCRITURA DEL CSV ─────────────────────────────────────────────────────────
with open(CSV_OUTPUT, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    for row in resultados:
        writer.writerow(row)

print(f'\nCSV generado: {CSV_OUTPUT}')
print(f'Total de textos clasificados: {len(resultados)}')
print('\nPrimeras 10 filas:')
print('Num, PAD, RNN, Transformer')
for row in resultados[:10]:
    print(','.join(str(x) for x in row))

### 4.3 Verificación del Formato del CSV

In [ ]:
# Verificamos que el CSV cumple el formato pedido: num,PRED_PAD,PRED_RNN,PRED_TF
print(f'Verificando formato de {CSV_OUTPUT}...')
with open(CSV_OUTPUT, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f'  Total de líneas : {len(lines)}')
ok = True
for i, line in enumerate(lines, 1):
    parts = line.strip().split(',')
    if len(parts) != 4:
        print(f'  ERROR en línea {i}: se esperaban 4 columnas, hay {len(parts)}')
        ok = False
    elif not parts[0].isdigit():
        print(f'  ERROR en línea {i}: primera columna no es número')
        ok = False
    elif not all(p in ('P', 'R') for p in parts[1:]):
        print(f'  ERROR en línea {i}: predicciones deben ser P o R')
        ok = False

if ok:
    print(f'  ✓ Formato correcto. Fichero listo para entregar.')
    print(f'  Nombre del fichero: alu{NUMERO_ALU}.csv')

---
## Guardar Vocabulario y Configuración para el Segundo Cuaderno

Guardamos todo lo necesario para poder cargar los modelos entrenados en el segundo cuaderno.

In [ ]:
import pickle

# Guardamos el vocabulario y la configuración de los modelos
checkpoint = {
    # Vocabulario PAD y Transformer (sin stopwords)
    'vocab_token_to_idx'     : vocab.token_to_idx,
    'vocab_idx_to_token'     : vocab.idx_to_token,
    'vocab_size'             : vocab.size,
    # Vocabulario RNN (con stopwords)
    'vocab_rnn_token_to_idx' : vocab_rnn.token_to_idx,
    'vocab_rnn_idx_to_token' : vocab_rnn.idx_to_token,
    'vocab_rnn_size'         : vocab_rnn.size,
    # Parámetros de preprocesado
    'context_size'           : CONTEXT_SIZE,
    'context_size_rnn'       : CONTEXT_SIZE_RNN,
    'min_freq'               : MIN_FREQ,
    # Arquitectura de cada modelo
    'pad_params'             : best_pad_params,
    'rnn_params'             : best_rnn_params,
    'tf_params'              : best_tf_params,
}

with open(f'{OUTPUT_DIR}/config.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)

print(f'Configuración guardada en {OUTPUT_DIR}/config.pkl')
print(f'Modelos guardados en:')
print(f'  {OUTPUT_DIR}/modelo_pad_final.pth')
print(f'  {OUTPUT_DIR}/modelo_rnn_final.pth')
print(f'  {OUTPUT_DIR}/modelo_transformer_final.pth')
print('\nDescarga estos archivos para usarlos en el segundo cuaderno.')